In [1]:
import calendar
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import create_engine
from pandas.tseries.offsets import BDay

engine = create_engine("sqlite:///c:\\ruby\\port_lite\\db\\development.sqlite3")
conlite = engine.connect()

engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

pd.set_option('display.max_row',None)

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"

today = date.today()
yesterday = today - timedelta(days=1)
today, yesterday

(datetime.date(2023, 1, 10), datetime.date(2023, 1, 9))

In [2]:
# specify the number of business days
num_days = 1
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(num_days)
yesterday = today - num_business_days
#yesterday = yesterday.date()
print(f'today: {today}')
print(f'yesterday: {yesterday}')

today: 2023-01-10
yesterday: 2023-01-09 00:00:00


In [3]:
yesterday = yesterday.date()
a_year_ago = yesterday - timedelta(days=365)
yesterday, a_year_ago

(datetime.date(2023, 1, 9), datetime.date(2022, 1, 9))

### Restart and Run All Cells

### Weekly process or when stataus changes.

In [4]:
pd.read_sql_query('SELECT COUNT(*) AS records FROM sales', conlite)

,records
0,1982


In [5]:
sqlDel = """DELETE FROM sales"""
rp = conlite.execute(sqlDel)
rp.rowcount

2303

In [6]:
sql = """
SELECT name
FROM orders 
ORDER BY name
"""
df = pd.read_sql(sql, conlite)

names = df['name'].values.tolist()
in_p = ", ".join(map(lambda name: "'%s'" % name, names))
in_p

"'ASP', 'BANPU', 'BCH', 'CKP', 'CPF', 'CPNREIT', 'GFPT', 'GVREIT', 'IVL', 'JMART', 'LPF', 'MAKRO', 'PTT', 'SCC', 'SINGER', 'SYNEX', 'TFFIF', 'TFG', 'TMT', 'WHAIR'"

In [7]:
len(in_p.split(','))

20

### Get past one year data

In [8]:
sql = """
SELECT * 
FROM price 
WHERE date >= '%s' AND name IN (%s) 
ORDER BY name, date"""
sql = sql % (a_year_ago, in_p)
print(sql)


SELECT * 
FROM price 
WHERE date >= '2022-01-09' AND name IN ('ASP', 'BANPU', 'BCH', 'CKP', 'CPF', 'CPNREIT', 'GFPT', 'GVREIT', 'IVL', 'JMART', 'LPF', 'MAKRO', 'PTT', 'SCC', 'SINGER', 'SYNEX', 'TFFIF', 'TFG', 'TMT', 'WHAIR') 
ORDER BY name, date


In [10]:
df = pd.read_sql(sql, const)
df.shape

(4840, 7)

In [11]:
data_path = "../data/"
file_name = "Yearly-Price-by-Name.csv"
output_file = data_path + file_name

df.set_index("name", inplace=True)
df.to_csv(output_file, header=None)

### Create monitors from orders

In [16]:
sql = """
SELECT name,trade 
FROM orders 
WHERE name IN (%s)
ORDER BY name
"""
sql = sql % in_p
orders = pd.read_sql(sql, conlite)
orders.set_index(['name'],inplace=True)
orders

,trade
name,
ASP,S
BANPU,S
BCH,S
CKP,B
CPF,B
CPNREIT,B
GFPT,B
GVREIT,B
IVL,B


In [17]:
file_name = "monitors.csv"
data_file = data_path + file_name
output_file = csv_path + file_name
box_file = box_path + file_name

orders.to_csv(data_file, header=None)
orders.to_csv(output_file)
orders.to_csv(box_file)

In [18]:
sql = """
SELECT trade, COUNT(*) AS items 
FROM orders
GROUP BY trade
ORDER BY trade
"""
grp = pd.read_sql(sql, conlite)
grp

,trade,items
0,B,13
1,S,7


### For new orders that never have XXX.html & XXX.CSV

In [19]:
file_name = 'price-uploads.csv'
url = data_path + file_name
url

'../data/price-uploads.csv'

In [20]:
uploads = pd.read_csv(url)
uploads.sort_values(['name'],ascending=[True]).shape

(145, 1)

In [21]:
df_merge = pd.merge(orders, uploads, on='name', how='outer', indicator=True)
df_merge.sort_values(['name'],ascending=[True]).shape

(149, 3)

In [22]:
new_prices = df_merge['_merge'] == 'left_only'
df_merge[new_prices]

,name,trade,_merge
4,CPF,B,left_only
5,CPNREIT,B,left_only
16,TFFIF,B,left_only
17,TFG,B,left_only


In [23]:
new_prices = df_merge['_merge'] == 'right_only'
df_merge[new_prices].shape

(129, 3)

In [24]:
sql = """
SELECT name, status, market
FROM stocks 
ORDER BY status, name
"""
stocks = pd.read_sql(sql, conlite)
stocks.set_index(["name"], inplace=True)
stocks.shape

(67, 2)

In [25]:
file_name = "stocks-all.csv"
data_file = data_path + file_name
output_file = csv_path + file_name
box_file = box_path + file_name
one_file = one_path + file_name

stocks.to_csv(data_file, header=None)
stocks.to_csv(output_file)
stocks.to_csv(box_file)
stocks.to_csv(one_file)

In [26]:
sql = """
SELECT name, status 
FROM stocks 
WHERE status IN ("B","O")
ORDER BY name
"""
buy_candidates = pd.read_sql(sql, conlite)
buy_candidates.set_index(["name"], inplace=True)
buy_candidates

,status
name,
CKP,O
CPF,O
CPNREIT,B
GFPT,O
GVREIT,B
IVL,B
JMART,O
LPF,O
PTT,O


In [27]:
buy_candidates.shape[0]

13

In [28]:
sql = """
SELECT name, status 
FROM stocks 
WHERE status IN ("I","S")
ORDER BY name
"""

sell_candidates = pd.read_sql(sql, conlite)
sell_candidates.set_index(["name"], inplace=True)
sell_candidates

,status
name,
ASP,S
BANPU,I
BCH,I
MAKRO,I
SCC,I
SYNEX,I
TMT,S


In [29]:
sell_candidates.shape[0]

7